In [1]:
import os, glob, gc, re
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pickle, warnings
import torch
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings('ignore')

# ── Paths ───────────────────────────────────────────────────
LAB_DIR     = "../data/lab"
EV_DIR      = "../data/ev_real_world"
RESULTS_DIR = "../results/metrics"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Hyperparams ─────────────────────────────────────────────
WINDOW      = 20
STRIDE      = 10        # ← skip every 10 cycles (reduces correlation)
EOL_SOH     = 0.80
MIN_CAP     = 0.3

FEATURE_COLS = ['cap_Ah','v_mean','v_min','v_slope','i_mean',
                't_mean','t_max','energy_Wh','duration_min',
                'soc_start','soc_end',
                'EFC_norm']         # ← relative position in life
N_FEATURES   = len(FEATURE_COLS)

# ── Stratified split — applied AFTER processing all cells ───
# (call stratified_cell_split(df_all) at end of Cell 4)
TRAIN_IDS, VAL_IDS, TEST_IDS = set(), set(), set()

def stratified_cell_split(df_all, seed=42):
    """Split cells by EoL bucket → all splits see same degradation speeds."""
    cell_eol = df_all.groupby('cell_id')['EoL_EFC'].first().reset_index()
    cell_eol = cell_eol[cell_eol['EoL_EFC'] <= 2000]       # drop censored
    cell_eol['bucket'] = pd.qcut(cell_eol['EoL_EFC'], q=4, labels=False)

    train_ids, temp = train_test_split(
        cell_eol['cell_id'], test_size=0.25,
        stratify=cell_eol['bucket'], random_state=seed)

    temp_buckets = cell_eol.set_index('cell_id').loc[temp, 'bucket']
    val_ids, test_ids = train_test_split(
        temp, test_size=0.5,
        stratify=temp_buckets, random_state=seed)

    return set(train_ids), set(val_ids), set(test_ids)

print("✅ Config loaded")
print(f"   Window: {WINDOW}  Stride: {STRIDE}  Features: {N_FEATURES}")


✅ Config loaded
   Window: 20  Stride: 10  Features: 12


In [2]:
def extract_cycle_features(df):
    """
    Extract one row of features per discharge cycle from raw 30s time-series.
    Returns DataFrame with cycle-level health indicators.
    """
    df = df.copy()
    # Label discharge segments
    df['is_discharge'] = df['i_raw_A'] < -0.5
    df['is_charge']    = df['i_raw_A'] >  0.5
    df['seg_id']       = (df['is_discharge'] != df['is_discharge'].shift()).cumsum()

    cycles = []
    discharge_segs = df[df['is_discharge']].groupby('seg_id')

    for seg_id, seg in discharge_segs:
        cap = abs(seg['delta_q_Ah'].iloc[-1] - seg['delta_q_Ah'].iloc[0])
        if cap < MIN_CAP:
            continue

        # Time in minutes
        duration = (seg['timestamp_s'].iloc[-1] - seg['timestamp_s'].iloc[0]) / 60.0

        # Voltage curve — linear slope (degradation signature)
        v = seg['v_raw_V'].values
        t = np.arange(len(v))
        v_slope = np.polyfit(t, v, 1)[0] if len(v) > 2 else 0.0

        # Energy delivered (Wh)
        dt_h = 30 / 3600  # 30s steps in hours
        energy = np.sum(seg['v_raw_V'].values * np.abs(seg['i_raw_A'].values)) * dt_h

        cycles.append({
            'EFC':             seg['EFC'].mean(),
            'cap_Ah':          round(cap, 4),
            'v_mean':          seg['v_raw_V'].mean(),
            'v_min':           seg['v_raw_V'].min(),
            'v_slope':         v_slope,
            'i_mean':          seg['i_raw_A'].mean(),
            't_mean':          seg['t_cell_degC'].mean(),
            't_max':           seg['t_cell_degC'].max(),
            'energy_Wh':       round(energy, 4),
            'duration_min':    round(duration, 2),
            'soc_start':       seg['soc_est'].iloc[0],
            'soc_end':         seg['soc_est'].iloc[-1],
        })

    return pd.DataFrame(cycles)

# ── Test on one cell ───────────────────────────────────────
test_f  = glob.glob(f"{LAB_DIR}/cell_log_age_30s_P067_1*.csv")[0]
df_raw  = pd.read_csv(test_f, sep=';')
df_cyc  = extract_cycle_features(df_raw)

print(f"✅ Cycle features extracted: {len(df_cyc)} cycles")
print(f"   EFC range:  {df_cyc['EFC'].min():.1f} → {df_cyc['EFC'].max():.1f}")
print(f"   Cap range:  {df_cyc['cap_Ah'].min():.3f} → {df_cyc['cap_Ah'].max():.3f} Ah")
print(f"\n{df_cyc.head(3).round(3).to_string()}")


✅ Cycle features extracted: 26 cycles
   EFC range:  0.1 → 206.3
   Cap range:  0.308 → 3.164 Ah

     EFC  cap_Ah  v_mean  v_min  v_slope  i_mean  t_mean   t_max  energy_Wh  duration_min  soc_start  soc_end
0  0.069   0.762   3.062  2.600   -0.029  -2.625  27.451  29.682      2.381          17.0     26.389    0.963
1  0.862   2.898   3.639  2.500   -0.003  -0.999  25.119  26.933     10.578         174.0     99.331    2.576
2  1.268   0.361   3.391  3.249   -0.005  -0.980  25.387  25.641      1.248          22.0     24.670   12.413


In [3]:
def fit_rul_labels(df_cyc, df_raw):
    """
    Fit quadratic degradation curve through RPT points.
    Assign RELATIVE RUL (0→1) to every discharge cycle.
    """
    rpt = df_raw[df_raw['cap_aged_est_Ah'].notna()][
        ['EFC','cap_aged_est_Ah']].copy()
    if len(rpt) < 2:
        return None

    rpt['SoH'] = rpt['cap_aged_est_Ah'] / rpt['cap_aged_est_Ah'].iloc[0]
    nominal    = rpt['cap_aged_est_Ah'].iloc[0]

    # Interpolate EoL at 80% SoH
    try:
        if (rpt['SoH'] <= EOL_SOH).any():
            f_eol   = interp1d(rpt['SoH'].values[::-1],
                               rpt['EFC'].values[::-1],
                               kind='linear', fill_value='extrapolate')
            eol_efc = float(f_eol(EOL_SOH))
        else:
            coeffs  = np.polyfit(rpt['EFC'], rpt['SoH'], 2)
            efc_ext = np.linspace(rpt['EFC'].max(), rpt['EFC'].max()*4, 3000)
            hits    = efc_ext[np.polyval(coeffs, efc_ext) <= EOL_SOH]
            eol_efc = float(hits[0]) if len(hits) > 0 else rpt['EFC'].max() * 2
    except:
        eol_efc = rpt['EFC'].max() * 1.5

    # Skip censored cells — EoL extrapolation too unreliable
    if eol_efc > 2000:
        return None

    # Fit quadratic SoH → per-cycle estimates
    coeffs = np.polyfit(rpt['EFC'], rpt['SoH'], 2)

    df_cyc = df_cyc.copy()
    df_cyc['SoH_est']  = np.polyval(coeffs, df_cyc['EFC']).clip(0, 1)
    df_cyc['cap_est']  = df_cyc['SoH_est'] * nominal
    df_cyc['EFC_norm'] = (df_cyc['EFC'] / eol_efc).clip(0, 1)   # ← relative age
    df_cyc['RUL']      = ((eol_efc - df_cyc['EFC']) / eol_efc).clip(0, 1)  # ← relative RUL 0→1
    df_cyc['EoL_EFC']  = eol_efc
    df_cyc['nominal']  = nominal

    return df_cyc

# ── Test on one cell ────────────────────────────────────────
df_labeled = fit_rul_labels(df_cyc, df_raw)
print(f"✅ RUL labels assigned: {len(df_labeled)} cycles")
print(f"   EFC_norm range: {df_labeled['EFC_norm'].min():.3f} → {df_labeled['EFC_norm'].max():.3f}")
print(f"   RUL range:      {df_labeled['RUL'].min():.3f} → {df_labeled['RUL'].max():.3f}  (relative, 0→1)")
print(f"   EoL EFC:        {df_labeled['EoL_EFC'].iloc[0]:.1f}")
print(f"\n{df_labeled[['EFC','EFC_norm','SoH_est','RUL']].head(5).round(3).to_string()}")


✅ RUL labels assigned: 26 cycles
   EFC_norm range: 0.000 → 1.000
   RUL range:      0.000 → 1.000  (relative, 0→1)
   EoL EFC:        138.9

     EFC  EFC_norm  SoH_est    RUL
0  0.069     0.000     0.99  1.000
1  0.862     0.006     0.99  0.994
2  1.268     0.009     0.99  0.991
3  1.851     0.013     0.99  0.987
4  1.952     0.014     0.99  0.986


In [6]:
from tqdm import tqdm

def process_all_lab_cells(lab_dir):
    files   = sorted(glob.glob(f"{lab_dir}/cell_log_age_30s_*.csv"))
    all_dfs = []
    skipped = []

    for fpath in tqdm(files, desc="Processing cells"):
        cell_id = os.path.basename(fpath).replace('.csv','')
        try:
            df_raw = pd.read_csv(fpath, sep=';')
            df_cyc = extract_cycle_features(df_raw)
            if len(df_cyc) < WINDOW + 1:
                skipped.append(cell_id)
                continue
            df_lab = fit_rul_labels(df_cyc, df_raw)
            if df_lab is None:          # None = censored cell (EoL>2000)
                skipped.append(cell_id)
                continue
            df_lab['cell_id'] = cell_id
            all_dfs.append(df_lab)
            del df_raw; gc.collect()
        except Exception as e:
            skipped.append(f"{cell_id}: {e}")

    df_all = pd.concat(all_dfs, ignore_index=True)
    print(f"\n✅ Processed: {len(all_dfs)} cells")
    print(f"⚠️  Skipped (censored + short): {len(skipped)}")

    # Apply stratified split here
    TRAIN_IDS, VAL_IDS, TEST_IDS = stratified_cell_split(df_all)
    df_all['split'] = df_all['cell_id'].apply(
        lambda x: 'train' if x in TRAIN_IDS
             else 'val'   if x in VAL_IDS
             else 'test'
    )

    for s in ['train','val','test']:
        sub = df_all[df_all['split']==s]
        print(f"  {s:5s}: {sub['cell_id'].nunique():3d} cells | "
              f"RUL mean={sub['RUL'].mean():.3f}  "
              f"min={sub['RUL'].min():.3f}  max={sub['RUL'].max():.3f}")

    return df_all

df_all = process_all_lab_cells(LAB_DIR)
df_all.to_parquet(f"{RESULTS_DIR}/lab_cycles_labeled.parquet", index=False)
print(f"\n💾 Saved → results/metrics/lab_cycles_labeled.parquet")


Processing cells: 100%|███████████████████████| 219/219 [05:52<00:00,  1.61s/it]



✅ Processed: 206 cells
⚠️  Skipped (censored + short): 13
  train: 154 cells | RUL mean=0.316  min=0.000  max=1.000
  val  :  26 cells | RUL mean=0.300  min=0.000  max=1.000
  test :  26 cells | RUL mean=0.297  min=0.000  max=1.000

💾 Saved → results/metrics/lab_cycles_labeled.parquet


In [11]:
# ── Cell 5 Notebook 02 — rerun with correct 12 features ────
FEATURE_COLS = ['cap_Ah','v_mean','v_min','v_slope','i_mean',
                't_mean','t_max','energy_Wh','duration_min',
                'soc_start','soc_end',
                'EFC_norm']          # ← add this line

train_df = df_all[df_all['split'] == 'train']

scaler_X = StandardScaler()
scaler_y = StandardScaler()

scaler_X.fit(train_df[FEATURE_COLS])
scaler_y.fit(train_df[['RUL']])

with open(f"{RESULTS_DIR}/scaler_X.pkl", 'wb') as f:
    pickle.dump(scaler_X, f)
with open(f"{RESULTS_DIR}/scaler_y.pkl", 'wb') as f:
    pickle.dump(scaler_y, f)

print(f"✅ Scalers refitted")
print(f"   n_features:  {scaler_X.n_features_in_}")   # must print 12
print(f"   RUL mean:    {scaler_y.mean_[0]:.4f}")      # relative scale ~0.3-0.5
print(f"   RUL std:     {scaler_y.scale_[0]:.4f}")     # relative scale ~0.3


✅ Scalers refitted
   n_features:  12
   RUL mean:    0.3165
   RUL std:     0.3354


In [12]:
class LabBatteryDataset(Dataset):
    """
    Sliding window LSTM dataset from KIT NMC lab cells.
    Each sample: (window of W cycles × F features) → RUL scalar
    """
    def __init__(self, df, scaler_X, scaler_y,
                 window=WINDOW, feature_cols=FEATURE_COLS):
        self.window   = window
        self.samples  = []

        for cell_id, grp in df.groupby('cell_id'):
            grp = grp.sort_values('EFC').reset_index(drop=True)
            X   = scaler_X.transform(grp[feature_cols].values)
            y   = scaler_y.transform(grp[['RUL']].values).flatten()

            for i in range(window, len(grp)):
                x_seq = X[i-window:i]       # (window, n_features)
                y_val = y[i]                 # scalar RUL
                self.samples.append((
                    torch.tensor(x_seq, dtype=torch.float32),
                    torch.tensor(y_val, dtype=torch.float32)
                ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


class EVBatteryDataset(Dataset):
    """
    PyTorch Dataset for EVBattery charging snippets.
    Each sample: (128 × 8 snippet) → capacity label (when available)
    Only loads snippets where capacity > 0.
    """
    def __init__(self, dataset_path, label_csv):
        self.samples  = []
        label_df      = pd.read_csv(label_csv, index_col=0)
        car_health    = dict(zip(label_df['car'], label_df['label']))

        pkl_files = sorted(glob.glob(f"{dataset_path}/data/*.pkl"))
        print(f"  Scanning {len(pkl_files):,} pkl files...")

        for fpath in tqdm(pkl_files[:10000], desc="Loading EV snippets"):
            try:
                data, meta = torch.load(fpath, map_location='cpu',
                                        weights_only=False)
                cap = meta.get('capacity', 0)
                if cap is None or cap == 0:
                    continue
                snippet = torch.tensor(data, dtype=torch.float32)  # (128,8)
                target  = torch.tensor(float(cap), dtype=torch.float32)
                health  = car_health.get(meta['car'], 0)
                self.samples.append((snippet, target, health))
            except:
                continue

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


print("✅ Dataset classes defined")
print("   LabBatteryDataset  → sliding window sequences")
print("   EVBatteryDataset   → charging snippet samples")


✅ Dataset classes defined
   LabBatteryDataset  → sliding window sequences
   EVBatteryDataset   → charging snippet samples


In [13]:
# ── Lab DataLoaders ─────────────────────────────────────────
train_ds = LabBatteryDataset(df_all[df_all['split']=='train'],
                              scaler_X, scaler_y)
val_ds   = LabBatteryDataset(df_all[df_all['split']=='val'],
                              scaler_X, scaler_y)
test_ds  = LabBatteryDataset(df_all[df_all['split']=='test'],
                              scaler_X, scaler_y)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=0)

# ── Quick shape verification ────────────────────────────────
xb, yb = next(iter(train_loader))
print(f"✅ Lab DataLoaders ready")
print(f"   Train batches: {len(train_loader):,}  |  batch X: {xb.shape}  y: {yb.shape}")
print(f"   Val   batches: {len(val_loader):,}")
print(f"   Test  batches: {len(test_loader):,}")
print(f"   Input shape per sample: ({WINDOW}, {len(FEATURE_COLS)})")
print(f"\n   Total train samples: {len(train_ds):,}")
print(f"   Total val   samples: {len(val_ds):,}")
print(f"   Total test  samples: {len(test_ds):,}")

print(f"\n🚀 Notebook 02 complete — DataLoaders ready for PI-LSTM training!")


✅ Lab DataLoaders ready
   Train batches: 2,458  |  batch X: torch.Size([64, 20, 12])  y: torch.Size([64])
   Val   batches: 461
   Test  batches: 362
   Input shape per sample: (20, 12)

   Total train samples: 157,262
   Total val   samples: 29,460
   Total test  samples: 23,153

🚀 Notebook 02 complete — DataLoaders ready for PI-LSTM training!
